# Machine Learning — Prévision des demandes d'asile à T+1

## Objectif
Prédire **`applied_t1`**, le nombre de demandes du même segment à l'année T+1.

Approches comparées :
1. Baseline naïve : `prediction = applied`
2. Ridge Regression
3. Random Forest Regressor
4. HistGradientBoosting Regressor

Le protocole est strictement temporel : **Train → Validation → Test futur**.
Le choix expérimental se fait sur Validation ; le Test sert à mesurer la généralisation finale.


In [ ]:
# 1. IMPORTS ET PARAMÈTRES
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, median_absolute_error,
    mean_squared_error, mean_squared_log_error
)

pd.set_option("display.max_columns", 100)

DATA_PATH = Path("dataset_ml_features_v3.csv")
RESULTS_PATH = Path("model_comparison.csv")
PREDICTIONS_PATH = Path("test_predictions.csv")


In [ ]:
# 2. CHARGEMENT
df = pd.read_csv(DATA_PATH)

print("Lignes :", len(df))
print("Colonnes :", df.shape[1])
print("Période :", int(df["year"].min()), "-", int(df["year"].max()))
display(df.head())


## 3. Cible et variables explicatives

La cible est `applied_t1`.  
`growth_t1` est exclu car il dépend de T+1 et provoquerait une fuite de cible.


In [ ]:
# 3. DÉFINITION DES FEATURES
TARGET = "applied_t1"
FORBIDDEN = ["applied_t1", "growth_t1"]

candidate_categorical = [
    "coo_id", "coa_id", "origin_region", "asylum_region",
    "procedure_type", "app_type", "dec_level", "app_pc"
]

categorical_features = [
    c for c in candidate_categorical if c in df.columns
]

feature_columns = [
    c for c in df.columns if c not in FORBIDDEN
]

numeric_features = [
    c for c in feature_columns if c not in categorical_features
]

assert TARGET not in feature_columns
assert "growth_t1" not in feature_columns

print("Catégorielles :", len(categorical_features))
print("Numériques :", len(numeric_features))
print("Total :", len(feature_columns))


In [ ]:
# 4. POPULATION D'APPRENTISSAGE
# Seules les lignes dont T+1 est connu peuvent être évaluées.
model_df = df[df[TARGET].notna()].copy()

# Les lignes récentes sans cible sont conservées à part pour un futur scoring.
scoring_df = df[df[TARGET].isna()].copy()

assert (model_df[TARGET] >= 0).all()

print("Avec cible :", len(model_df))
print("Sans cible :", len(scoring_df))


# 5. Split temporel

Pas de `train_test_split` aléatoire.

- toutes les années anciennes : **Train**
- avant-dernière année disponible : **Validation**
- dernière année disponible : **Test**

Cela reproduit le fonctionnement réel : apprendre sur le passé et prédire le futur.


In [ ]:
# 5. CONSTRUCTION DU SPLIT TEMPOREL
years = sorted(model_df["year"].dropna().unique())
assert len(years) >= 3, "Au moins 3 années avec cible sont nécessaires."

validation_year = years[-2]
test_year = years[-1]

train_df = model_df[model_df["year"] < validation_year].copy()
validation_df = model_df[model_df["year"] == validation_year].copy()
test_df = model_df[model_df["year"] == test_year].copy()

assert len(train_df) > 0 and len(validation_df) > 0 and len(test_df) > 0

print("Train :", int(train_df.year.min()), "-", int(train_df.year.max()), "|", len(train_df))
print("Validation :", int(validation_year), "|", len(validation_df))
print("Test :", int(test_year), "|", len(test_df))


In [ ]:
# 6. SÉPARATION X / y
X_train = train_df[feature_columns].copy()
y_train = train_df[TARGET].copy()

X_val = validation_df[feature_columns].copy()
y_val = validation_df[TARGET].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[TARGET].copy()

print("Train :", X_train.shape)
print("Validation :", X_val.shape)
print("Test :", X_test.shape)


## 7. Préprocessing sans leakage

Les imputations, standardisations et encodages sont contenus dans des `Pipeline`.  
Ils sont donc appris uniquement pendant `.fit(X_train, y_train)`.

- Numériques : médiane.
- Ridge : médiane + standardisation.
- Catégorielles : `UNKNOWN` + One-Hot Encoding.
- Catégories nouvelles : ignorées proprement avec `handle_unknown="ignore"`.


In [ ]:
# 7. PRÉPROCESSING
numeric_ridge = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

ridge_preprocessor = ColumnTransformer([
    ("num", numeric_ridge, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

tree_preprocessor = ColumnTransformer([
    ("num", numeric_tree, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])


In [ ]:
# 8. FONCTION D'ÉVALUATION
def evaluate(y_true, y_pred):
    # Les volumes négatifs n'ont pas de sens métier.
    pred = np.clip(np.asarray(y_pred), 0, None)

    return {
        "MAE": mean_absolute_error(y_true, pred),
        "MedAE": median_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "RMSLE": np.sqrt(mean_squared_log_error(y_true, pred))
    }


# 9. Baseline naïve

Hypothèse : **le volume de T+1 sera identique au volume de T**.

C'est la référence minimale que le ML doit améliorer.


In [ ]:
# 9. BASELINE
predictions = {}
metrics = {}

for split_name, X, y in [
    ("Train", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Test", X_test, y_test)
]:
    pred = X["applied"].to_numpy()
    predictions[("Baseline", split_name)] = pred
    metrics[("Baseline", split_name)] = evaluate(y, pred)

display(pd.DataFrame([metrics[("Baseline", "Validation")]]))


# 10. Ridge Regression

Ridge teste si les relations entre historique, contexte et T+1 peuvent être captées par une structure relativement simple et régularisée.


In [ ]:
# 10. RIDGE
ridge = Pipeline([
    ("preprocessing", ridge_preprocessor),
    ("model", Ridge(alpha=1.0))
])

ridge.fit(X_train, y_train)

for split_name, X, y in [
    ("Train", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Test", X_test, y_test)
]:
    pred = ridge.predict(X)
    predictions[("Ridge", split_name)] = pred
    metrics[("Ridge", split_name)] = evaluate(y, pred)

display(pd.DataFrame([metrics[("Ridge", "Validation")]]))


# 11. Random Forest Regressor

Random Forest teste l'apport des relations non linéaires et des interactions entre variables.  
Les paramètres restent volontairement modérés pour cette première comparaison.


In [ ]:
# 11. RANDOM FOREST
random_forest = Pipeline([
    ("preprocessing", tree_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

random_forest.fit(X_train, y_train)

for split_name, X, y in [
    ("Train", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Test", X_test, y_test)
]:
    pred = random_forest.predict(X)
    predictions[("Random Forest", split_name)] = pred
    metrics[("Random Forest", split_name)] = evaluate(y, pred)

display(pd.DataFrame([metrics[("Random Forest", "Validation")]]))


# 12. HistGradientBoosting Regressor

Cette première expérimentation utilise les **variables numériques** pour HistGradientBoosting.  
Cela évite de densifier une matrice One-Hot potentiellement très volumineuse. Les catégories pourront être intégrées ensuite si cette famille de modèles montre un signal intéressant.


In [ ]:
# 12. HISTGRADIENTBOOSTING
hgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        random_state=42
    ))
])

hgb.fit(X_train[numeric_features], y_train)

for split_name, X, y in [
    ("Train", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Test", X_test, y_test)
]:
    pred = hgb.predict(X[numeric_features])
    predictions[("HistGradientBoosting", split_name)] = pred
    metrics[("HistGradientBoosting", split_name)] = evaluate(y, pred)

display(pd.DataFrame([metrics[("HistGradientBoosting", "Validation")]]))


# 13. Comparaison sur Validation

C'est cette table qui sert à comparer les approches avant de regarder la généralisation finale sur Test.


In [ ]:
# 13. TABLEAU GLOBAL DES MÉTRIQUES
rows = []
for (model, split), values in metrics.items():
    rows.append({"Model": model, "Split": split, **values})

results = pd.DataFrame(rows)
results.to_csv(RESULTS_PATH, index=False)

validation_results = (
    results[results["Split"] == "Validation"]
    .sort_values("MAE")
    .copy()
)

baseline_val_mae = metrics[("Baseline", "Validation")]["MAE"]
validation_results["MAE_gain_vs_baseline_pct"] = (
    (baseline_val_mae - validation_results["MAE"])
    / baseline_val_mae * 100
)

display(validation_results.round(3))


# 14. Généralisation sur Test

Le Test représente la dernière année disponible.  
Il ne doit pas servir à modifier les hyperparamètres après consultation répétée.


In [ ]:
# 14. RÉSULTATS TEST
test_results = (
    results[results["Split"] == "Test"]
    .sort_values("MAE")
    .copy()
)

baseline_test_mae = metrics[("Baseline", "Test")]["MAE"]
test_results["MAE_gain_vs_baseline_pct"] = (
    (baseline_test_mae - test_results["MAE"])
    / baseline_test_mae * 100
)

display(test_results.round(3))


# 15. Diagnostic d'overfitting

Nous comparons MAE Train et MAE Test.  
Un écart très important signale une généralisation temporelle à examiner.


In [ ]:
# 15. DIAGNOSTIC TRAIN / TEST
overfit_rows = []

for model in ["Baseline", "Ridge", "Random Forest", "HistGradientBoosting"]:
    train_mae = metrics[(model, "Train")]["MAE"]
    test_mae = metrics[(model, "Test")]["MAE"]

    overfit_rows.append({
        "Model": model,
        "Train_MAE": train_mae,
        "Test_MAE": test_mae,
        "Test_Train_MAE_Ratio": test_mae / train_mae if train_mae > 0 else np.nan
    })

overfit_report = pd.DataFrame(overfit_rows)
display(overfit_report.sort_values("Test_MAE").round(3))


# 16. Prédictions détaillées du Test

Le fichier permet d'analyser les erreurs par corridor plutôt que de regarder uniquement une moyenne globale.


In [ ]:
# 16. EXPORT DES PRÉDICTIONS
test_predictions = test_df[
    ["year", "coo_id", "coa_id", "applied", TARGET]
].copy()

test_predictions["Baseline"] = np.clip(
    predictions[("Baseline", "Test")], 0, None
)
test_predictions["Ridge"] = np.clip(
    predictions[("Ridge", "Test")], 0, None
)
test_predictions["Random_Forest"] = np.clip(
    predictions[("Random Forest", "Test")], 0, None
)
test_predictions["HistGradientBoosting"] = np.clip(
    predictions[("HistGradientBoosting", "Test")], 0, None
)

test_predictions.to_csv(PREDICTIONS_PATH, index=False)
display(test_predictions.head())


In [ ]:
# 17. VISUALISATION RÉEL VS PRÉDIT
for model in ["Ridge", "Random_Forest", "HistGradientBoosting"]:
    plt.figure(figsize=(7, 6))
    plt.scatter(
        test_predictions[TARGET],
        test_predictions[model],
        alpha=0.35
    )

    maximum = max(
        test_predictions[TARGET].max(),
        test_predictions[model].max()
    )

    # La diagonale correspond à une prédiction parfaite.
    plt.plot([0, maximum], [0, maximum], linestyle="--")
    plt.xlabel("applied_t1 réel")
    plt.ylabel("applied_t1 prédit")
    plt.title(f"Réel vs prédit — {model}")
    plt.tight_layout()
    plt.show()


# 18. Analyse par taille de corridor

Une métrique globale peut masquer de fortes erreurs sur les corridors importants.  
Les observations du Test sont donc réparties en quatre groupes selon `applied` à T.


In [ ]:
# 18. ERREUR PAR TAILLE DE CORRIDOR
test_predictions["corridor_size"] = pd.qcut(
    test_predictions["applied"],
    q=4,
    duplicates="drop"
)

rows = []

for model in ["Baseline", "Ridge", "Random_Forest", "HistGradientBoosting"]:
    for size, group in test_predictions.groupby("corridor_size", observed=True):
        rows.append({
            "Model": model,
            "Corridor_Size": str(size),
            "N": len(group),
            "MAE": mean_absolute_error(group[TARGET], group[model])
        })

error_by_size = pd.DataFrame(rows)
display(error_by_size.sort_values(["Corridor_Size", "MAE"]).round(3))


# 19. Règles d'interprétation

Un modèle n'est pas retenu uniquement parce qu'il affiche le plus petit MAE.

Il faut vérifier simultanément :

- son gain par rapport à la baseline ;
- sa stabilité entre Validation et Test ;
- l'écart Train/Test ;
- ses performances sur les grands corridors ;
- la concentration éventuelle des grosses erreurs sur certains pays ou segments.

Si les modèles complexes n'améliorent pas suffisamment la baseline, la priorité sera de réexaminer le signal, la granularité et les features plutôt que d'ajouter immédiatement du tuning.
